In [1]:
import os
import torch
import json 
import re
import ast

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from collections import defaultdict
from torch_geometric.data import Data, HeteroData, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj
from torch_geometric.nn import to_hetero
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm
from collections import Counter

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


# Load sub-graph data

In [2]:
strict = True
isco = True

if isco:
    if strict:
        df_graphs = pd.read_parquet("../kg_construction/subgraphs_isco_no_inference_strict.parquet")
    else:
        df_graphs = pd.read_parquet("../kg_construction/subgraphs_isco_no_inference_easy.parquet")
else:
    if strict:
        df_graphs = pd.read_parquet("../kg_construction/subgraphs_no_inference_strict.parquet")
    else:
        df_graphs = pd.read_parquet("../kg_construction/subgraphs_no_inference_easy.parquet")

df_graphs.head()

,isco,model,prompt,vacancy,cvid,response,graph,num_nodes,num_edges,density,avg_degree,avg_clustering,path_exists,shortest_path_length
0,True,qwen,structured,1527549.0,1ea333bcf408489d88d6bb4ecad3852d,0,"{""directed"": false, ""multigraph"": false, ""grap...",67,72,0.032564,2.149254,0.0,True,2
1,True,qwen,structured,1527507.0,6f38a6778f1249b5b14658aed49f5348,0,"{""directed"": false, ""multigraph"": false, ""grap...",45,41,0.041414,1.822222,0.0,True,3
2,True,qwen,structured,1527507.0,238e44044b5348e09dba220756613ac6,0,"{""directed"": false, ""multigraph"": false, ""grap...",64,73,0.036210,2.281250,0.0,True,3
3,True,qwen,structured,1527507.0,6e5a9a5cac7e454bb17c85853850b108,0,"{""directed"": false, ""multigraph"": false, ""grap...",56,62,0.040260,2.214286,0.0,True,3
4,True,qwen,structured,1527507.0,2772752bd7fb4fdebd3c2cdbf7539ee7,1,"{""directed"": false, ""multigraph"": false, ""grap...",44,47,0.049683,2.136364,0.0,True,3


In [3]:
df_graphs["response"].value_counts()

response
0    107734
1     10077
Name: count, dtype: int64

In [4]:
non_zero = df_graphs.groupby("cvid")["response"].sum() > 0
non_zero = set(non_zero[non_zero == True].index)
len(non_zero), len(df_graphs.groupby("cvid"))

df_graphs = df_graphs[df_graphs["cvid"].isin(non_zero)]

In [5]:
truth_dict = df_graphs.groupby('cvid').apply(
    lambda x: dict(zip(x['vacancy'], x['response']))
).to_dict()

C:\Users\roans\AppData\Local\Temp\ipykernel_24288\1297208639.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  truth_dict = df_graphs.groupby('cvid').apply(


# Create starting embeddings

In [6]:
G = nx.DiGraph()

with open("../outputs/inferred_outputs/kg_qwen_structured.edgelist", encoding="utf-8") as f:
    for line in tqdm(f.readlines()):
        
        if "err:error" in line:
            print(line)
            continue
            
        s, p, o = eval(line)
        G.add_edge(s, o, edge_type=p)

edge_type_dict = nx.get_edge_attributes(G, 'edge_type')
unique_types = list(edge_type_dict.values())

  0%|          | 0/2671845 [00:00<?, ?it/s]

In [10]:
hits = defaultdict(lambda : defaultdict(lambda : defaultdict(lambda : defaultdict(lambda : defaultdict))))
misses = defaultdict(lambda : defaultdict(lambda : defaultdict(lambda : defaultdict(lambda : defaultdict))))

for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:

        relevant = df_graphs[(df_graphs["model"] == model) & (df_graphs["prompt"] == prompt)]
        
        for row in tqdm(relevant.iterrows(), total=len(relevant)):
            if not pd.isna(row[1][6]):    
                graph_json_string = row[1][6]
                
                # Convert string to dict
                data_dict = json.loads(graph_json_string)
                g = nx.node_link_graph(data_dict)
            
                if row[1][5] == 0:
                    misses[model][prompt][row[1][4]][row[1][3]] = g
                else:
                    hits[model][prompt][row[1][4]][row[1][3]] = g

  0%|          | 0/7552 [00:00<?, ?it/s]

C:\Users\roans\AppData\Local\Temp\ipykernel_24288\2656502039.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not pd.isna(row[1][6]):
C:\Users\roans\AppData\Local\Temp\ipykernel_24288\2656502039.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  graph_json_string = row[1][6]
C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\networkx\readwrite\json_graph\node_link.py:287: FutureWarning: 
The default value will be changed to `edges="edges" in NetworkX 3.6.

To make this warning go away, explicitly set the edges kwarg, e.g.:

  nx.node_link_graph(data, edges="links") to preserve current behavior, o

  0%|          | 0/7682 [00:00<?, ?it/s]

  0%|          | 0/7658 [00:00<?, ?it/s]

  0%|          | 0/7731 [00:00<?, ?it/s]

  0%|          | 0/7732 [00:00<?, ?it/s]

  0%|          | 0/2739 [00:00<?, ?it/s]

  0%|          | 0/3793 [00:00<?, ?it/s]

  0%|          | 0/2929 [00:00<?, ?it/s]

  0%|          | 0/162 [00:00<?, ?it/s]

In [11]:
emb_size = 30

label_types = {i : torch.rand(emb_size) for i in set(unique_types)}

# Filter data

## Load dajobbert

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Embedding using: {device}")

# 2. Move Model to GPU
tokenizer = AutoTokenizer.from_pretrained("../pipeline/dajobbert-kg-specialized")
embed_model = AutoModel.from_pretrained("../pipeline/dajobbert-kg-specialized").to(device)
embed_model.eval() # Set to evaluation mode

def get_bert_embedding(text):
    # 3. Move Inputs to GPU
    inputs = tokenizer(text, return_tensors="pt", padding=True, 
                       truncation=True, max_length=64).to(device)
    
    with torch.no_grad():
        outputs = embed_model(**inputs)
    
    # 4. Move Result back to CPU for NetworkX storage
    return outputs.last_hidden_state[0, 0, :].detach().cpu()

Embedding using: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

# Create dataloaders

In [13]:
ALLOWED_NODES = {
    "JOB_TITLE", "SKILL", "QUALITY", "EXPERIENCE_LEVEL", 
    "WORK_EXPERIENCE", "EDUCATION_LEVEL", "CERTIFICATION", 
    "LOCATION", "CONTRACT_TYPE", "INDUSTRY", "LANGUAGE", 
    "COMPANY", "SALARY", "CANDIDATE", "VACANCY"
}

ALLOWED_EDGES = {
    "REQUIRES_SKILL", "REQUIRES_QUALITY", "DESIRES", 
    "REQUIRES_EXPERIENCE_LEVEL", "REQUIRES_WORK_EXPERIENCE", 
    "REQUIRES_EDUCATION", "REQUIRES_CERTIFICATION", 
    "REQUIRES_LANGUAGE", "INVOLVES_TASK", "OFFERS_POSITION", 
    "HAS_LOCATION", "HAS_CONTRACT_TYPE", "HAS_SALARY_RANGE", 
    "IS_MANAGER_LEVEL", "IS_IN_INDUSTRY"
}

def clean_node_name(name):
    """Strips URI junk, prefixes, and standardizes to lowercase string."""
    return str(name).replace("jie:", "").replace("_jie_", "").replace("xsd:", "").strip("_").lower()

def get_node_base_type_strict(node_str):
    """Maps cleaned node names to our schema or discards as None."""
    node_up = node_str.upper()
    if "CANDIDATE" in node_up: return "candidate"
    if "POSITION" in node_up or "VACANCY" in node_up: return "vacancy"
    
    for allowed in ALLOWED_NODES:
        if allowed in node_up:
            return allowed.lower()
    return None

In [14]:
SCHEMA_NODES = [
    "job_title", "skill", "quality", "experience_level", 
    "work_experience", "education_level", "certification", 
    "location", "contract_type", "industry", "language", 
    "company", "salary", "candidate", "vacancy", "miscellaneous"
]

SCHEMA_TRIPLETS = [
    ('candidate', 'gets_recommended', 'vacancy'),
    ('candidate', 'has_skill', 'skill'),
    ('candidate', 'has_experience', 'work_experience'),
    ('candidate', 'has_education', 'education_level'),
    ('candidate', 'lives_in', 'location'),
    ('candidate', 'speaks_language', 'language'),
    ('candidate', 'has_job_title', 'job_title'),
    ('vacancy', 'requires_skill', 'skill'),
    ('vacancy', 'requires_quality', 'quality'),
    ('vacancy', 'requires_education', 'education_level'),
    ('vacancy', 'requires_experience_level', 'experience_level'),
    ('vacancy', 'has_location', 'location'),
    ('vacancy', 'offers_position', 'job_title'),
    ('vacancy', 'has_salary_range', 'salary'),
    ('job_title', 'is_in_industry', 'industry'),
    ('company', 'offers_position', 'vacancy'),
    ('candidate', 'related_to', 'miscellaneous'),
    ('vacancy', 'related_to', 'miscellaneous')
]
for ntype in SCHEMA_NODES: SCHEMA_TRIPLETS.append((ntype, 'self_loop', ntype))

In [15]:
def build_embedding_cache(hits_misses, batch_size=64):
    """Embeds all unique nodes in one massive GPU-accelerated batch."""
    unique_names = set()
    print("Collecting unique node names...")
    for graphs in hits_misses.values():
        for g in graphs.values():
            for node in g.nodes():
                c_n = deep_clean(node)
                if not (c_n.isdigit() or c_n in ['decimal', 'integer', 'string', '']):
                    unique_names.add(c_n)
    
    name_list = list(unique_names)
    cache = {}
    print(f"Embedding {len(name_list)} unique nodes on {device}...")
    
    for i in tqdm(range(0, len(name_list), batch_size)):
        batch_names = name_list[i : i + batch_size]
        inputs = tokenizer(batch_names, return_tensors="pt", padding=True, 
                           truncation=True, max_length=64).to(device)
        
        with torch.no_grad():
            outputs = embed_model(**inputs)
        
        # [CLS] token (index 0) for each name in the batch
        embeddings = outputs.last_hidden_state[:, 0, :].cpu()
        
        for name, emb in zip(batch_names, embeddings):
            cache[name] = emb
            
    return cache

In [16]:
def deep_clean(name):
    name = str(name).lower()
    name = re.sub(r'jie:|xsd:|_jie_|_jip_', '', name)
    name = re.sub(r'_\d+$', '', name)
    return name.strip('_').replace('_', ' ').strip()

def get_ntype(node_str):
    raw = str(node_str).upper()
    if "CANDIDATE" in raw: return "candidate"
    if "POSITION" in raw or "VACANCY" in raw: return "vacancy"
    for a in SCHEMA_NODES:
        if a.upper().replace('_', ' ') in raw: return a
    return "miscellaneous"

def create_dataloaders(hits_misses, truth_dict, embedding_cache, 
                       train_size=0.8, val_size=0.1):
    
    embedding_dim = embed_model.config.hidden_size
    train_list, val_list, test_list = [], [], []
    u_keys = list(hits_misses.keys())
    t_idx, v_idx = int(len(u_keys)*train_size), int(len(u_keys)*(train_size+val_size))

    for i, user_id in tqdm(enumerate(u_keys), total=len(u_keys)):
        target_list = train_list if i < t_idx else (val_list if i < v_idx else test_list)
        graphs = hits_misses[user_id]
        
        temp_graphs, truth_values = [], []
        node_info, head_ids, tail_ids = {}, [], []
        global_node_counter = 1

        for sg_idx, (tail_id, G_raw) in enumerate(graphs.items()):
            if not G_raw: continue
            
            # Find Anchors
            cand_node, vac_node = None, None
            for n in list(G_raw.nodes()):
                c_n = deep_clean(n)
                if str(user_id) in c_n and "candidate" in c_n: cand_node = n
                if str(tail_id) in c_n and ("position" in c_n or "vacancy" in c_n): vac_node = n

            if cand_node and vac_node:
                G_raw.add_edge(cand_node, vac_node, edge_type="gets_recommended")
                undirected = G_raw.to_undirected()
                if nx.has_path(undirected, cand_node, vac_node):
                    main_comp = nx.node_connected_component(undirected, cand_node)
                    G_raw = G_raw.subgraph(main_comp).copy()
                else: continue

            valid_nodes_in_sg = []
            for n in G_raw.nodes():
                c_n = deep_clean(n)
                # Look up from CACHE instead of running BERT
                if c_n in embedding_cache:
                    ntype = get_ntype(n)
                    new_name = f"{n}_{i}_{sg_idx}"
                    node_info[new_name] = (ntype, global_node_counter, sg_idx)
                    
                    G_raw.nodes[n]['x'] = embedding_cache[c_n]
                    
                    if ntype == "candidate" and str(user_id) in c_n:
                        head_ids.append(global_node_counter)
                    elif (ntype == "vacancy" or ntype == "job_title") and str(tail_id) in c_n:
                        tail_ids.append(global_node_counter)
                    
                    valid_nodes_in_sg.append((n, new_name))
                    global_node_counter += 1

            if valid_nodes_in_sg:
                mapping = {old: new for old, new in valid_nodes_in_sg}
                sg_final = G_raw.subgraph([n for n, _ in valid_nodes_in_sg])
                temp_graphs.append(nx.relabel_nodes(sg_final, mapping))
                
                u_truth = truth_dict.get(user_id, truth_dict.get(str(user_id), {}))
                label = u_truth.get(tail_id, u_truth.get(int(tail_id), 0))
                truth_values.append(float(label))

        # Build HeteroData
        merged_G = nx.compose_all(temp_graphs)
        data = HeteroData()
        node_map = {} 

        type_data = defaultdict(lambda: {"x":[], "u":[], "s":[]})
        for node in merged_G.nodes():
            ntype, uid, sg = node_info[node]
            node_map[node] = len(type_data[ntype]["x"])
            type_data[ntype]["x"].append(merged_G.nodes[node]['x'])
            type_data[ntype]["u"].append(uid)
            type_data[ntype]["s"].append(sg)

        for ntype in SCHEMA_NODES:
            if ntype in type_data:
                data[ntype].x = torch.stack(type_data[ntype]["x"])
                data[ntype].unique_node_id = torch.tensor(type_data[ntype]["u"])
                data[ntype].sub_graph = torch.tensor(type_data[ntype]["s"])
            else:
                data[ntype].x = torch.zeros((1, embedding_dim))
                data[ntype].unique_node_id = torch.tensor([0])
                data[ntype].sub_graph = torch.tensor([-1])

        # Edges and Self-loops (Standard Logic)
        for triplet in SCHEMA_TRIPLETS:
            data[triplet].edge_index = torch.empty((2, 0), dtype=torch.long)

        for u, v, d in merged_G.edges(data=True):
            u_t, _, _ = node_info[u]; v_t, _, _ = node_info[v]
            etype = deep_clean(d.get("edge_type", "related_to")).replace(' ', '_')
            triplet = (u_t, etype, v_t)
            if triplet in [ (t[0], t[1], t[2]) for t in SCHEMA_TRIPLETS]:
                new_e = torch.tensor([[node_map[u]], [node_map[v]]], dtype=torch.long)
                data[triplet].edge_index = torch.cat([data[triplet].edge_index, new_e], dim=1)

        for ntype in data.node_types:
            num_n = data[ntype].x.size(0)
            idx = torch.arange(num_n)
            data[ntype, "self_loop", ntype].edge_index = torch.stack([idx, idx], dim=0)

        data.y, data.head_nodes, data.tail_nodes = torch.tensor(truth_values), torch.tensor(head_ids), torch.tensor(tail_ids)
        target_list.append(data)
        
    return train_list, val_list, test_list

In [17]:
if strict:
    s = "strict"
else:
    s = "easy"

for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        hits_curr = {user: {job: hits[model][prompt][user][job] for job in hits[model][prompt][user]} 
                     for user in hits[model][prompt]}        
        
        misses_curr = {user: {job: misses[model][prompt][user][job] for job in misses[model][prompt][user]} 
                       for user in misses[model][prompt]}       
        
        hits_misses_curr = {user: {**hits[model][prompt][user], **misses[model][prompt][user]}
                            for user in {**hits[model][prompt], **misses[model][prompt]}.keys() 
                            if (user in hits[model][prompt]) and (user in misses[model][prompt])} 
        
        # Build the cache once
        cache = build_embedding_cache(hits_misses_curr)
        
        # Create/load dataloaders
        train_loader, val_loader, test_loader = create_dataloaders(hits_misses_curr, truth_dict, cache)
        
        ### Batching not possible due to the fact that different graphs have different numbers of edge for each type - Considering the RAM usage, we did not implement this with, e.g., padding. 
        trainloader = DataLoader(train_loader) 
        valloader = DataLoader(val_loader) 
        testloader = DataLoader(test_loader)

        if isco:
            torch.save(trainloader, f'../dataloaders/{model}_{prompt}_isco_{s}_trainloader_no_inference.pth')
            torch.save(valloader, f'../dataloaders/{model}_{prompt}_isco_{s}_valloader_no_inference.pth')
            torch.save(testloader, f'../dataloaders/{model}_{prompt}_isco_{s}_testloader_no_inference.pth')
        else:
            torch.save(trainloader, f'../dataloaders/{model}_{prompt}_{s}_trainloader_no_inference.pth')
            torch.save(valloader, f'../dataloaders/{model}_{prompt}_{s}_valloader_no_inference.pth')
            torch.save(testloader, f'../dataloaders/{model}_{prompt}_{s}_testloader_no_inference.pth')

Embedding 21568 unique nodes on cuda...


  0%|          | 0/337 [00:00<?, ?it/s]

  0%|          | 0/365 [00:00<?, ?it/s]

Embedding 23826 unique nodes on cuda...


  0%|          | 0/373 [00:00<?, ?it/s]

...

  0%|          | 0/233 [00:00<?, ?it/s]

  0%|          | 0/218 [00:00<?, ?it/s]

Embedding 679 unique nodes on cuda...


  0%|          | 0/11 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]